# Lecture 22 — Implementing Transformers

**CMU 10-414/714, Deep Learning Systems (lecture by Zico Kolter)**
Companion notebook · Sangram Lembe

Kolter writes self-attention, multi-head attention and a full transformer layer in NumPy, checking each
against PyTorch. Here each fast version is checked against a slow, obviously-correct reference:

1. self-attention with one fused $W_{KQV}$ matrix, plus a causal mask
2. mini-batches **batch-first**, and why that needs a true *batch* matrix multiply
3. multi-head attention by reshaping and swapping axes — no loops
4. the full transformer layer

In [1]:
import numpy as np
rng = np.random.default_rng(0)

def softmax(Z):
    Z = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return Z / Z.sum(axis=-1, keepdims=True)

## 1. Self-attention, the way libraries store it

PyTorch keeps the three projections in one $D\times 3D$ block and splits the result — the same trick as the
LSTM's four gates. An output projection $W_{out}$ follows. Masks are **added** to the scores.

In [2]:
def self_attention(X, mask, W_KQV, W_out):
    K, Q, V = np.split(X @ W_KQV, 3, axis=-1)
    A = softmax(K @ Q.swapaxes(-1, -2) / np.sqrt(X.shape[-1]) + mask)
    return A @ V @ W_out, A

def attention_reference(X, mask, W_KQV, W_out):
    """Row by row, no matrix tricks."""
    D = X.shape[1]
    W_K, W_Q, W_V = W_KQV[:, :D], W_KQV[:, D:2*D], W_KQV[:, 2*D:]
    out = []
    for i in range(X.shape[0]):
        scores = np.array([(X[i] @ W_K) @ (X[j] @ W_Q) / np.sqrt(D) + mask[i, j]
                           for j in range(X.shape[0])])
        w = np.exp(scores - scores.max()); w /= w.sum()
        out.append(sum(w[j] * (X[j] @ W_V) for j in range(X.shape[0])) @ W_out)
    return np.array(out)

T, D = 10, 8
X = rng.normal(size=(T, D))
W_KQV, W_out = rng.normal(size=(D, 3 * D)) / np.sqrt(D), rng.normal(size=(D, D)) / np.sqrt(D)
M = np.triu(-np.inf * np.ones((T, T)), 1)              # causal mask
Y, A = self_attention(X, M, W_KQV, W_out)
print("matrix version == row-by-row reference:", np.allclose(Y, attention_reference(X, M, W_KQV, W_out)))

matrix version == row-by-row reference: True


## 2. Mini-batches: batch first

LSTMs put time first because they step through time. Attention multiplies whole $T\times D$ matrices, so it
wants each example's matrix contiguous: **batch first**, shape $(B, T, D)$.

Two different kinds of "batched" multiply appear, and only one is new:

In [3]:
B = 5
Xb = rng.normal(size=(B, T, D))
W = rng.normal(size=(D, D))

# (a) every example times the SAME weight: secretly one ordinary 2-D matmul
shared = Xb @ W
as_2d = (Xb.reshape(B * T, D) @ W).reshape(B, T, D)
print("(a) X @ W  ==  reshape to 2-D, multiply, reshape back:", np.allclose(shared, as_2d))

# (b) every example times ITS OWN matrix: a true batch matmul (BMM)
K, Q = rng.normal(size=(B, T, D)), rng.normal(size=(B, T, D))
bmm = K @ Q.swapaxes(-1, -2)                           # (B, T, T)
loop = np.stack([K[b] @ Q[b].T for b in range(B)])
print("(b) K @ Q^T batched  ==  loop over examples:", np.allclose(bmm, loop))
print("    note: Q.T would reverse ALL axes ->", Q.T.shape, "; swapaxes(-1,-2) ->", Q.swapaxes(-1, -2).shape)

(a) X @ W  ==  reshape to 2-D, multiply, reshape back: True
(b) K @ Q^T batched  ==  loop over examples: True
    note: Q.T would reverse ALL axes -> (8, 10, 5) ; swapaxes(-1,-2) -> (5, 8, 10)


Case (a) is what convolutions and MLPs did all along. Case (b) — $K_iQ_i^{\mathsf T}$ for each example $i$ —
has no 2-D equivalent; it needs a batch matrix multiply. And `.T` is wrong on 3-D arrays: use `swapaxes`.
The self-attention function above already handles batches unchanged:

In [4]:
Yb, _ = self_attention(Xb, M, W_KQV, W_out)
print("batched == each example alone:",
      all(np.allclose(Yb[b], self_attention(Xb[b], M, W_KQV, W_out)[0]) for b in range(B)))

batched == each example alone: True


## 3. Multi-head attention

Split the $D$ columns of $K$, $Q$, $V$ into $H$ groups and run attention separately in each, so different
heads can attend to different things. The fast way: reshape $(B,T,D) \to (B,T,H,D/H)$, swap to
$(B,H,T,D/H)$, run batched attention, swap back, reshape.

In [5]:
def multihead_attention(X, mask, heads, W_KQV, W_out):
    B, T, D = X.shape
    K, Q, V = np.split(X @ W_KQV, 3, axis=-1)
    split = lambda Z: Z.reshape(B, T, heads, D // heads).swapaxes(1, 2)     # (B, H, T, D/H)
    K, Q, V = split(K), split(Q), split(V)
    A = softmax(K @ Q.swapaxes(-1, -2) / np.sqrt(D // heads) + mask)        # (B, H, T, T)
    Y = (A @ V).swapaxes(1, 2).reshape(B, T, D)                              # back to (B, T, D)
    return Y @ W_out, A

def multihead_reference(X, mask, heads, W_KQV, W_out):
    """Loop over examples and heads explicitly."""
    B, T, D = X.shape
    dh = D // heads
    out = np.zeros((B, T, D))
    for b in range(B):
        K, Q, V = np.split(X[b] @ W_KQV, 3, axis=-1)
        cols = []
        for h in range(heads):
            s = slice(h * dh, (h + 1) * dh)
            A = softmax(K[:, s] @ Q[:, s].T / np.sqrt(dh) + mask)
            cols.append(A @ V[:, s])
        out[b] = np.concatenate(cols, axis=1) @ W_out
    return out

Ym, Am = multihead_attention(Xb, M, 4, W_KQV, W_out)
print("attention matrices:", Am.shape, " (batch, heads, T, T)")
print("reshape/swapaxes version == explicit loops:",
      np.allclose(Ym, multihead_reference(Xb, M, 4, W_KQV, W_out)))

attention matrices: (5, 4, 10, 10)  (batch, heads, T, T)
reshape/swapaxes version == explicit loops: True


Kolter's aside: PyTorch's multi-head layer returns the attention weights **averaged over heads**, which hides
exactly what makes the heads different. Here all $H$ matrices are kept.

## 4. The transformer layer

Multi-head attention, a residual connection and layer norm, then a two-layer ReLU network on each position,
with another residual and layer norm — what PyTorch calls `TransformerEncoderLayer`.

In [6]:
def layer_norm(Z, eps=1e-5):
    return (Z - Z.mean(-1, keepdims=True)) / np.sqrt(Z.var(-1, keepdims=True) + eps)

def transformer(X, mask, heads, W_KQV, W_out, W_ff1, W_ff2, eps=1e-5):
    Z = layer_norm(X + multihead_attention(X, mask, heads, W_KQV, W_out)[0], eps)
    return layer_norm(Z + np.maximum(0, Z @ W_ff1) @ W_ff2, eps)

W_ff1, W_ff2 = rng.normal(size=(D, 32)) / np.sqrt(D), rng.normal(size=(32, D)) / np.sqrt(32)
out = transformer(Xb, M, 4, W_KQV, W_out, W_ff1, W_ff2)
print("transformer layer:", Xb.shape, "->", out.shape)

X2 = Xb.copy(); X2[:, 7] += 5.0                           # change the input at time 7
out2 = transformer(X2, M, 4, W_KQV, W_out, W_ff1, W_ff2)
print("causal: outputs before t=7 unchanged:", np.allclose(out[:, :7], out2[:, :7]))
print("every position normalised: mean ~0, var ~1:",
      np.allclose(out.mean(-1), 0, atol=1e-6), np.allclose(out.var(-1), 1, atol=1e-3))

transformer layer: (5, 10, 8) -> (5, 10, 8)
causal: outputs before t=7 unchanged: True
every position normalised: mean ~0, var ~1: True True


## Summary

- Fused $W_{KQV}$ plus a split equals three separate projections; the matrix form matched a row-by-row reference.
- Batch-first suits attention. Multiplying by a shared weight is secretly 2-D; $K_iQ_i^{\mathsf T}$ per example
  needs a true batch matmul, and `swapaxes` rather than `.T`.
- Multi-head attention is a reshape and an axis swap — identical to looping over heads.
- A full transformer layer is about five lines on top, and stays causal under the mask.